<a href="https://colab.research.google.com/github/kamalrawat77/agentic-iam-lab/blob/main/week07-agentic-rag/Nugget038_ReAct_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nugget 038: Build a True ReAct Agent

In [10]:
!pip install -q google-genai
import json

In [106]:
from google import genai

def callGPT(prompt):
  client = genai.Client(api_key="APIKEY")
  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )

  return response

In [17]:
def cleanse_response(response):
  clean_response=clean_response = response.text
  clean_response = clean_response.replace("```json", "")
  clean_response = clean_response.replace("```", "")
  clean_response = clean_response.strip()

  return clean_response

In [18]:
def return_json(clean_response,objname):
  responseObj = json.loads(clean_response)
  if not objname:
    return responseObj
  resJsonObj  = responseObj[objname]
  return resJsonObj



Tool List

In [14]:
def dormant_accounts():

    return "Dormant Accounts: 27"


def department_breakdown():

    return """
IT: 45
HR: 12
Finance: 18
"""


def trend_analysis():
    return "Dormant accounts increased from 20 to 27"

def search_history():
    return "Past investigation found delayed terminations"

Tool Registry

In [24]:
tools = {
    "dormant_accounts": dormant_accounts,
    "department_breakdown": department_breakdown,
    "search_history": search_history,
    "trend_analysis": trend_analysis
}

Create memory list

In [73]:
memory = []

In [74]:
question = """
Why are dormant accounts increasing?
"""

Planner prompt

In [75]:
planner_prompt = f"""
You are an IAM investigator.

Question:
{question}

Current Memory:
{memory}

Available Tools:

- trend_analysis
- search_history

Decide next action.

Return JSON:

{{
 "action":"tool_name"
}}
"""

In [76]:
response=callGPT(planner_prompt)

In [77]:
toolDecision=return_json(cleanse_response(response),None)

In [78]:
print(toolDecision)

{'action': 'trend_analysis'}


In [79]:
tool_name = toolDecision["action"]

result = tools[tool_name]()

In [80]:
memory.append({
    "tool": tool_name,
    "observation": result
})

In [81]:
print(memory)

[{'tool': 'trend_analysis', 'observation': 'Dormant accounts increased from 20 to 27'}]


Run planner again with updated memory

In [86]:
planner_prompt = f"""
Question:
{question}

Memory:
{memory}

Available Tools:

- trend_analysis
- search_history

Decide next action based on memory if more evidence is needed for anwsering the question.

If enough evidence exists:

{{
  "action":"finish"
}}

Otherwise choose a tool as below
Return JSON:

{{
 "action":"tool_name"
}}
"""

In [87]:
print(planner_prompt)


Question:

Why are dormant accounts increasing?


Memory:
[{'tool': 'trend_analysis', 'observation': 'Dormant accounts increased from 20 to 27'}]

Available Tools:

- trend_analysis
- search_history

Decide next action based on memory if more evidence is needed for anwsering the question.

If enough evidence exists:

{
  "action":"finish"
}

Otherwise choose a tool as below
Return JSON:

{
 "action":"tool_name"
}



In [84]:
response=callGPT(planner_prompt)



In [85]:
print(response.text)

{
 "action":"trend_analysis"
}


Define a funtion for toolcalling

In [94]:
#Tool Registery and Tool definitions required before calling this
#Prompt must return toolname as per below format where toolLabel is the object to access the toolname
#{{
#  "toolLabel":"finish"
#}}

def callToolandUpdateMemory(prompt,toolLabel):
  response=callGPT(prompt)
  print(response.text)
  toolDecision=return_json(cleanse_response(response),None)
  print(toolDecision)
  tool_name = toolDecision[toolLabel]
  if "finish" == tool_name:
    return "No further action"
  result = tools[tool_name]()
  memory.append({
    "tool": tool_name,
    "observation": result
  })
  return result


In [89]:
result=callToolandUpdateMemory(planner_prompt,"action")

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""{
 "action":"search_history"
}"""
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-2.5-flash' prompt_feedback=None response_id='R289atXeJ7jWg8UPme-rwAY' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=11,
  prompt_token_count=119,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=119
    ),
  ],
  thoughts_token_count=152,
  total_token_count=282
) automatic_function_calling_history=[] parsed=None
{'action': 'search_history'}


In [91]:
print(result)

Past investigation found delayed terminations


In [90]:
print(memory)

[{'tool': 'trend_analysis', 'observation': 'Dormant accounts increased from 20 to 27'}, {'tool': 'search_history', 'observation': 'Past investigation found delayed terminations'}]


Run planner again with updated memory

In [95]:
planner_prompt = f"""
Question:
{question}

Memory:
{memory}

Available Tools:

- trend_analysis
- search_history

Decide next action based on memory if more evidence is needed for anwsering the question.

If enough evidence exists:

{{
  "action":"finish"
}}

Otherwise choose a tool as below
Return JSON:

{{
 "action":"tool_name"
}}
"""

In [96]:
result=callToolandUpdateMemory(planner_prompt,"action")

{
  "action":"finish"
}
{'action': 'finish'}


In [97]:
print(result)

No further action


Retrieve final answer based on updated memory

In [98]:
finalPrompt = f"""
Question:
{question}

Memory:
{memory}


"""

In [99]:
print(callGPT(finalPrompt).text)

Based on the information from your memory:

Dormant accounts are increasing because **past investigations found delayed terminations**.

This suggests that accounts that should have been closed or deactivated are remaining open for longer than intended. This delay could be due to various reasons, such as:

*   **Administrative Backlog:** A high volume of requests or a lack of resources to process terminations efficiently.
*   **Procedural Requirements:** Complex or time-consuming steps required to officially close an account.
*   **System Limitations:** Inefficient or outdated systems that hinder the automated identification and closure of truly dormant accounts.
*   **Verification Challenges:** Difficulty in verifying the true status of an account or contacting the account holder for confirmation.

While "delayed terminations" is the specific reason identified in your memory, other common factors that contribute to increasing dormant accounts in general include users forgetting about 

Lets create an actual loop

In [109]:
memory = []

Define helper functions

In [105]:
def planner(toolLabel):
  planner_prompt = f"""
  Question:
  {question}

  Memory:
  {memory}

  Available Tools:

  - trend_analysis
  - search_history

  Decide next action based on memory if more evidence is needed for anwsering the question.

  If enough evidence exists:

  {{
    "action":"finish"
  }}

  Otherwise choose a tool as below
  Return JSON:

  {{
  "action":"tool_name"
  }}
  """

  response=callGPT(planner_prompt)
  toolDecision=return_json(cleanse_response(response),None)
  tool_name = toolDecision[toolLabel]
  return tool_name

def run_tool(action):
  return {
    "tool": action,
    "observation": tools[action]()
  }

def answer():
  finalPrompt = f"""
  Question:
  {question}

  Memory:
  {memory}


  """

  print(callGPT(finalPrompt).text)




Run the loop

In [110]:
itr=1
while True:

    decision = planner("action")

    if decision == "finish":
        answer()
        break

    result = run_tool(decision)

    memory.append(result)
    print(f"Iteration {itr}")
    print(decision)
    print(memory)
    itr=itr+1

Iteration 1
trend_analysis
[{'tool': 'trend_analysis', 'observation': 'Dormant accounts increased from 20 to 27'}]
Iteration 2
search_history
[{'tool': 'trend_analysis', 'observation': 'Dormant accounts increased from 20 to 27'}, {'tool': 'search_history', 'observation': 'Past investigation found delayed terminations'}]
The increase in dormant accounts, from 20 to 27 (a 35% rise), can be attributed to several interconnected factors, with **delayed termination processes** being a key contributor as identified in past investigations.

Here's a breakdown of the reasons:

1.  **Delayed Termination Processes by Financial Institutions:**
    *   **Internal Policies & Operational Friction:** Banks often have specific protocols and waiting periods before an inactive account can be officially closed and its funds escheated (transferred to state unclaimed property). This process can be slow and resource-intensive.
    *   **Regulatory Requirements:** There are often strict regulations surroundin